# 02 — Análisis Exploratorio de Datos (EDA)

Explora distribuciones, correlaciones y patrones en las 2 453 AGEBs
de la Ciudad de México con todas las features físicas disponibles.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import setup_logging, ensure_output_dirs, FEATURE_COLS_RAW, TARGET_COL
from src.data_loader import load_ageb_v3
from src.preprocessing import create_target_variable, engineer_features
from src.visualization import (
    plot_correlation_matrix, plot_target_distribution, plot_feature_distributions
)

setup_logging()
ensure_output_dirs()
sns.set_theme(style='whitegrid')

## 1. Carga y preparación

In [ ]:
df = load_ageb_v3()
df = create_target_variable(df)
df = engineer_features(df)
print(f'{len(df)} AGEBs, {df.shape[1]} columnas')

## 2. Distribución del target

In [ ]:
plot_target_distribution(df, TARGET_COL, 'alto_riesgo')
print('Distribución de alto_riesgo:')
print(df['alto_riesgo'].value_counts())

## 3. Distribuciones de features

In [ ]:
plot_feature_distributions(df, FEATURE_COLS_RAW)

## 4. Matriz de correlación

In [ ]:
plot_correlation_matrix(df, FEATURE_COLS_RAW)

## 5. Correlaciones con el target

In [ ]:
corr_target = df[FEATURE_COLS_RAW + ['ruse_emergencias', 'alto_riesgo']].corr()['alto_riesgo'].drop('alto_riesgo').sort_values(ascending=False)
print('Correlación con alto_riesgo:')
for feat, val in corr_target.items():
    print(f'  {feat:40s} {val:+.4f}')

## 6. Análisis por tipo de suelo

In [ ]:
suelo_map = {1: 'Roca', 2: 'Mixto', 3: 'Arenoso'}
df['tipo_suelo_cat'] = df['tipo_suelo'].map(suelo_map).fillna('Desconocido')
print(df.groupby('tipo_suelo_cat')['ruse_emergencias'].describe())

## 7. Resumen de hallazgos EDA

In [ ]:
print('='*55)
print('  RESUMEN EDA')
print('='*55)
print(f'  Total AGEBs          : {len(df)}')
print(f'  Features físicas     : {len(FEATURE_COLS_RAW)}')
print(f'  Alto riesgo (target) : {df["alto_riesgo"].sum()} ({df["alto_riesgo"].mean():.1%})')
print(f'  Umbral P75           : {df[TARGET_COL].quantile(0.75):.1f} emergencias')
print('='*55)